# ДЗ. Повышение accuracy на MNIST

Старинская Светлана, 3ФИИТ

### 1. Импорт библиотек, константы и выбор устройства

Фиксируем seed для повторяемости. На Mac используется MPS, если он доступен; иначе CPU. Исходная accuracy из практики — 97,58%.

In [ ]:
import random
from copy import deepcopy

from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchvision import datasets, transforms

SEED = 42
EPOCHS = 8
BATCH_SIZE = 128
BASELINE_ACCURACY = 0.9758

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Устройство: {device}')

### 2. Подготовка данных

Нормализуем изображения MNIST и увеличиваем размер батча с 4 до 128, чтобы обучение было быстрее и стабильнее.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_data = datasets.MNIST('./mnist', train=True, download=True, transform=transform)
val_data = datasets.MNIST('./mnist', train=False, download=True, transform=transform)

loader_args = {'batch_size': BATCH_SIZE, 'num_workers': 0, 'pin_memory': device.type == 'cuda'}
train_loader = DataLoader(train_data, shuffle=True, generator=torch.Generator().manual_seed(SEED), **loader_args)
val_loader = DataLoader(val_data, shuffle=False, **loader_args)

### 3. Улучшенная нейросеть

Полносвязную сеть заменяем на CNN: свёртки учитывают структуру изображения. BatchNorm ускоряет обучение, Dropout уменьшает переобучение.

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.10),
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.15),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.30), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = MnistCNN().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

### 4. Базовые функции

Вынесем тестирование и отрисовку прогресса в отдельные функции: во время обучения будут видны loss и accuracy на train/test.

In [ ]:
@torch.inference_mode()
def evaluate(model, dataloader):
    model.eval()
    total_loss = correct = total = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

def show_progress(history):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], marker='o', label='train')
    axes[0].plot(epochs, history['val_loss'], marker='o', label='test')
    axes[0].set(title='Loss', xlabel='Эпоха', ylabel='Loss')
    axes[1].plot(epochs, history['train_accuracy'], marker='o', label='train')
    axes[1].plot(epochs, history['val_accuracy'], marker='o', label='test')
    axes[1].set(title='Accuracy', xlabel='Эпоха', ylabel='Accuracy', ylim=(0.8, 1.0))
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend()
    plt.tight_layout()
    plt.show()

### 5. Обучение и тестирование

Внутри эпохи отображается progress bar. После каждой эпохи обновляются графики и выводятся текущие метрики; сохраняются веса с лучшей test accuracy.

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}
best_accuracy, best_weights = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = train_correct = train_total = 0
    progress = tqdm(train_loader, desc=f'Эпоха {epoch}/{EPOCHS}', leave=False)
    for images, labels in progress:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)
        progress.set_postfix(loss=f'{loss.item():.3f}')

    val_loss, val_accuracy = evaluate(model, val_loader)
    train_accuracy = train_correct / train_total
    history['train_loss'].append(train_loss / train_total)
    history['val_loss'].append(val_loss)
    history['train_accuracy'].append(train_accuracy)
    history['val_accuracy'].append(val_accuracy)
    if val_accuracy > best_accuracy:
        best_accuracy, best_weights = val_accuracy, deepcopy(model.state_dict())
    scheduler.step()

    clear_output(wait=True)
    show_progress(history)
    lr = optimizer.param_groups[0]['lr']
    print(f'Эпоха {epoch}/{EPOCHS} | train accuracy: {train_accuracy:.2%} | test accuracy: {val_accuracy:.2%} | lr: {lr:.5f}')

model.load_state_dict(best_weights)
print(f'Исходная accuracy: {BASELINE_ACCURACY:.2%}')
print(f'Лучшая accuracy: {best_accuracy:.2%}')

### 6. Финальное сравнение и примеры предсказаний

Сравниваем accuracy исходной и улучшенной модели и смотрим несколько предсказаний. Графики обучения остаются в выводе предыдущего блока.

In [ ]:
values = [BASELINE_ACCURACY, best_accuracy]
plt.figure(figsize=(6, 4))
plt.bar(['Исходная', 'Улучшенная'], values, color=['gray', 'seagreen'])
plt.title('Сравнение accuracy')
plt.ylabel('Accuracy')
plt.ylim(0.9, 1.0)
for index, value in enumerate(values):
    plt.text(index, value + 0.002, f'{value:.2%}', ha='center')
plt.tight_layout()
plt.show()

images, labels = next(iter(val_loader))
with torch.inference_mode():
    predictions = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for image, label, prediction, axis in zip(images[:10], labels[:10], predictions[:10], axes.ravel()):
    axis.imshow(image.squeeze() * 0.3081 + 0.1307, cmap='gray')
    axis.set_title(f'{label} → {prediction}', color='seagreen' if label == prediction else 'crimson')
    axis.axis('off')
plt.tight_layout()
plt.show()